# BUSI70575 — Meta-Model Submission

**Team:** Sreeram_experimental. **Date:** 2026-06-04. **Brief:** [hm-ai.github.io/BUSI70575](https://hm-ai.github.io/BUSI70575/coursework/).

This notebook reproduces the rubric narrative end-to-end against the **cached pipeline outputs** committed under `results/sreeram_experimental/`. The full pipeline that produced those outputs is in `src/stml/experimental/` and is invoked stage-by-stage per `SUBMISSION_README.md` §3 — a single run takes ~60 minutes on CPU.

**Deliverables** (under `outputs/`):
- `metamodel_predictions.csv` — Required. H1 2022, full (date × instrument) grid, `prediction ∈ [0, 1]`.
- `strategy_weights.csv` — Optional bonus. Same grid, signed position weight.

**How this notebook is organised** (one section per rubric item):
1. Setup & inventory.
2. **§1 Feature engineering** (20 marks).
3. **§2 Triple-barrier labels** (20 marks).
4. **§3 Model development & comparison** (30 marks).
5. **§4 Cluster-level feature importance** (10 marks).
6. **§5 Model evaluation** (20 marks).
7. **Strategy construction** (+10 bonus).
8. **Deliverable CSV verification.**


## Section 0 — Setup & inventory

In [ ]:
from __future__ import annotations
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

DATA = ROOT / "data"
RESULTS = ROOT / "results" / "sreeram_experimental"
OUTPUTS = ROOT / "outputs"

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.dpi"] = 110

print(f"repo root: {ROOT}")
print(f"data:    {DATA.relative_to(ROOT)}/  ({sum(1 for _ in DATA.rglob('*') if _.is_file())} files)")
print(f"results: {RESULTS.relative_to(ROOT)}/  ({sum(1 for _ in RESULTS.rglob('*') if _.is_file())} files)")
print(f"outputs: {OUTPUTS.relative_to(ROOT)}/  ({sum(1 for _ in OUTPUTS.glob('*.csv') if _.is_file())} CSVs)")

In [ ]:
# Brief: 11 instruments across three asset classes.
ohlcv = pd.read_csv(DATA / "ohlcv_data.csv", parse_dates=["date"])
signals = pd.read_csv(DATA / "primary_signals.csv", parse_dates=["date"])

INSTRUMENTS = [c for c in signals.columns if c != "date"]
ASSET_CLASS = {
    "es1s": "Equity", "nq1s": "Equity", "fesx1s": "Equity",
    "cl1s": "Energy", "ho1s": "Energy", "rb1s": "Energy", "ng1s": "Energy",
    "gc1s": "Metals", "si1s": "Metals", "hg1s": "Metals", "pl1s": "Metals",
}
INST_NAME = {
    "es1s": "ES (S&P 500)", "nq1s": "NQ (Nasdaq 100)", "fesx1s": "FESX (Euro Stoxx)",
    "cl1s": "CL (WTI Crude)", "ho1s": "HO (Heating Oil)", "rb1s": "RB (RBOB Gasoline)",
    "ng1s": "NG (Natural Gas)", "gc1s": "GC (Gold)", "si1s": "SI (Silver)",
    "hg1s": "HG (Copper)", "pl1s": "PL (Platinum)",
}

print(f"Released window: {signals['date'].min().date()} -> {signals['date'].max().date()} "
      f"({len(signals)} trading days)")
print(f"Instruments ({len(INSTRUMENTS)}): {', '.join(INSTRUMENTS)}")
print()
print("Per-asset-class signal balance over the released window:")
for cls in ["Equity", "Energy", "Metals"]:
    insts = [i for i in INSTRUMENTS if ASSET_CLASS[i] == cls]
    long_n = sum((signals[i] == 1).sum() for i in insts)
    short_n = sum((signals[i] == -1).sum() for i in insts)
    flat_n = sum((signals[i] == 0).sum() for i in insts)
    total = long_n + short_n + flat_n
    print(f"  {cls:7s} ({len(insts)} inst): long {long_n:5d} ({100*long_n/total:.0f}%) | "
          f"short {short_n:5d} ({100*short_n/total:.0f}%) | flat {flat_n:5d} ({100*flat_n/total:.0f}%)")

## Section 1 — Feature engineering (rubric §1, 20 marks)

We ship **105 features across 18 families**. Eighteen because we keep the conventional `F1..F11` price/microstructure/macro families and add Bloomberg-augmented blocks `F12..F19` for cross-asset, term-structure, options-implied vol, EIA inventory, and PIT-lagged macro. Full taxonomy + per-feature justification in `reports/feature-catalog.md`; the code lives in `src/stml/experimental/features/`.

**Leakage discipline.** Two classes:
- **E (engineered)** — no fit; causal by truncation-invariance.
- **TF (fitted)** — GMM/HMM regimes, PCA/KMeans, macro z-scorer, drift discriminator — fit on the **FE-train block only** (`date ≤ 2021-07-01`) and applied causally with frozen parameters.

**Standardisation.** Every scale-dependent E-class column ships a parallel `z_<col>` twin — per-instrument causal expanding-window z-score with `min_periods=60`. Bounded / already-normalized columns (ratios, probabilities, t-stats, sin/cos) get no twin.

**Bloomberg augmentation (R-11).** Five cleaned parquets under `data/bloomberg/cleaned/` carry PIT-aligned futures term structure, options IV, EIA inventory, copper stocks, and the macro panel. Publication lags applied: daily series lag 1 day, weekly EIA lag 5 calendar days, monthly PMI lag 1 business day after month-end. The shipped model was selected to be robust to BBG missingness; see `results/sreeram_experimental/bbg_missingness_ablation.csv` for the per-class AUC delta with vs. without BBG (R-11 risk acknowledgement).

In [ ]:
# Bloomberg ingestion provenance.
bbg_clean_dir = DATA / "bloomberg" / "cleaned"
if bbg_clean_dir.exists():
    bbg_files = sorted(bbg_clean_dir.glob("*.parquet"))
    print(f"Bloomberg cleaned parquets ({len(bbg_files)}):")
    for p in bbg_files:
        size_kb = p.stat().st_size / 1024
        print(f"  {p.name:30s}  {size_kb:7.0f} KB")

# BBG missingness ablation (R-11 — pre-submission decision evidence).
ablation_path = RESULTS / "bbg_missingness_ablation.csv"
if ablation_path.exists():
    print("\nBBG-missingness ablation (AUC with vs without BBG, per asset class):")
    ablation = pd.read_csv(ablation_path)
    display(ablation.round(4))

## Section 2 — Triple-barrier labels (rubric §2, 20 marks)

Per-instrument barrier geometry `(pt, sl, h)` picked by an adjusted-Sharpe grid search (343 configurations) on the development partition only — the test partition `(2021-12-31 → 2022-06-29)` was never touched during barrier selection. The methodology is in `notebooks/jay/triple-barrier-label.pdf`; the per-instrument winners and search statistics in `results/sreeram_experimental/jay_geometry_summary.csv`.

The geometry varies across instruments because the underlying return distributions and signal frequencies do:
- **Thin signals (cl1s, fesx1s, ho1s, ng1s, nq1s, pl1s)** prefer narrow symmetric barriers (`pt = sl = 0.25 σ`) with `h = 1`. This converges to a *next-day signed-return* label — which the EDA shows is what the primary signal predicts on those instruments (lead/lag peaks at `k = +1`).
- **Equity indices (es1s, hg1s)** prefer `pt = 1.0 σ`, `sl = 0.25 σ`, `h = 10`. The 4:1 asymmetry reflects the right-skew of the primary signal's PnL on these — a small stop, a wider take-profit, ten-day window.
- **Metals (gc1s, si1s, pl1s)** use longer windows (`h = 15`–`20`); `rb1s` uses the most asymmetric (`2.5σ / 0.25σ`) reflecting its highly trending tail behaviour.

The labels CSV ships with an authoritative `partition` column (`train` / `val` / `test`) computed once on the released window — the pipeline honours it everywhere downstream.

In [ ]:
labels = pd.read_csv(DATA / "triple_barrier_labels.csv", parse_dates=["date", "t1"])
print(f"Total labelled events: {len(labels):,}")
print(f"  partition counts: {labels['partition'].value_counts().to_dict()}")
print(f"  test partition window: {labels.loc[labels['partition']=='test', 'date'].min().date()} -> "
      f"{labels.loc[labels['partition']=='test', 'date'].max().date()}")
print()

# Per-instrument geometry.
rows = []
for inst in INSTRUMENTS:
    sub = labels[labels['instrument'] == inst]
    rows.append({
        'instrument': INST_NAME[inst],
        'asset_class': ASSET_CLASS[inst],
        'n_events': len(sub),
        'pt': sub['pt'].iloc[0],
        'sl': sub['sl'].iloc[0],
        'h': sub['h'].iloc[0],
        'label_1_share': round(sub['label'].mean(), 3),
        'touch_pt_pct': round((sub['touch'] == 'pt').mean() * 100, 0),
        'touch_sl_pct': round((sub['touch'] == 'sl').mean() * 100, 0),
        'touch_vert_pct': round((sub['touch'] == 'vert').mean() * 100, 0),
    })
geometry = pd.DataFrame(rows)
display(geometry)

## Section 3 — Model development & comparison (rubric §3, 30 marks)

Five model families compared per instrument under **CPCV(6, 2) with 1-SE rule**:
- **Linear:** elastic-net logistic regression (L1+L2 grid, balanced class weights)
- **Tree:** Random Forest (bootstrap sample weights = AFML Ch. 4 average uniqueness)
- **Boosting:** XGBoost, LightGBM (early-stopping on inner-fold AUC)
- **Neural:** multi-task MLP with per-instrument heads + shared embedding (S4)

**Champion selection** uses Harry's `INSTRUMENT_REGIMES` pool taxonomy — per instrument, evaluate the candidate (pool, model) combinations and pick the one with the highest **lower-1-SE CPCV AUC**. This is more conservative than picking the highest mean AUC: it rewards stable models over flukier high-mean / high-variance ones.

In [ ]:
champions = pd.read_csv(RESULTS / "champions_summary.csv")
champions.insert(1, "name", [INST_NAME.get(i, i) for i in champions['instrument']])
display_cols = [c for c in champions.columns if c in {
    'instrument', 'name', 'variant', 'winning_pool', 'winning_model',
    'champion_auc', 'champion_sem', 'lower_ci_1se', 'selection_reason',
}]
print("Per-instrument champion (CPCV(6,2) + 1-SE rule):")
display(champions[display_cols].round(4))

# Per-class summary.
per_class = pd.read_csv(RESULTS / "baseline_xgb_per_class.csv")
print("\nPer-class baseline AUCs:")
display(per_class.round(4))

In [ ]:
# Per-pool per-model breakdown — the comparison table the rubric asks for.
pool_breakdown_path = RESULTS / "champions_per_pool_per_model.csv"
if pool_breakdown_path.exists():
    pool = pd.read_csv(pool_breakdown_path)
    print("Full per-pool per-model AUC breakdown (top rows):")
    display(pool.head(20).round(4))

## Section 4 — Cluster-level feature importance (rubric §4, 10 marks)

We cluster the feature matrix per asset class via a **Mantegna correlation tree** (`scipy.cluster.hierarchy.linkage`, `ward`, distance threshold tuned by gap statistic) and compute importance **at the cluster level**, not just per-feature:
- **MDA** (Mean Decrease in Accuracy) — permute every feature in the cluster jointly, measure CPCV-AUC drop
- **MDI** (gain-based importance from XGBoost) summed over cluster members
- **SHAP** magnitude summed over cluster members

All three are cross-checked via Kendall-τ rank agreement (in `rank_agreement.csv`). The 'deep' run additionally fits a pruned variant per asset class (keeping only the significant clusters) and verifies that pruned-vs-full AUC delta is small.

**Bug fixes documented:** four importance-pipeline bugs found and fixed during S5; see `reports/sreeram_experimental/s4_s5_summary.md` for the audit trail (TreeSHAP via native `pred_contribs`, permutation seed propagation, cluster-id stability under fold variance, partition-respecting CPCV inside the importance loop).

In [ ]:
# Deep-importance summary across asset classes.
deep = pd.read_csv(RESULTS / "importance" / "deep_summary.csv")
print("Pruned-vs-full feature variant per asset class:")
display(deep.round(4))

# Per-class crosscheck table (MDA / MDI / SHAP rank agreement).
for cls in ["equity", "energy", "metals"]:
    p = RESULTS / "importance" / cls / "cluster_crosscheck_table.csv"
    if p.exists():
        cct = pd.read_csv(p)
        sig = cct[cct.get('significant', False) == True] if 'significant' in cct.columns else cct.head(5)
        print(f"\n{cls.upper()} — significant clusters by MDA:")
        display(sig.head(8).round(4))

## Section 5 — Model evaluation (rubric §5, 20 marks)

Per the brief: classification metrics, confusion-matrix / decision-threshold analysis, per-instrument breakdown, and a primary-blind baseline comparison.

**Classification metrics — per instrument, on the test partition (H1 2022):**

In [ ]:
baseline = pd.read_csv(RESULTS / "baseline_per_instrument.csv")
baseline.insert(1, "name", [INST_NAME.get(i, i) for i in baseline['instrument']])
cols = ['instrument', 'name', 'asset_class', 'n', 'pos_rate', 'auc',
        'precision', 'recall', 'f1', 'log_loss', 'brier']
cols = [c for c in cols if c in baseline.columns]
print("Per-instrument classification metrics (vs primary-blind baseline):")
display(baseline[cols].round(4))

In [ ]:
# Decision-threshold analysis — bootstrap p* = L / (G + L) from train returns (slide 21).
thr_path = RESULTS / "threshold_summary.csv"
if thr_path.exists():
    thr = pd.read_csv(thr_path)
    if 'instrument' in thr.columns:
        thr.insert(1, 'name', [INST_NAME.get(i, i) for i in thr['instrument']])
    print("Per-instrument bootstrap p* threshold (from train-fold TP/FP returns):")
    display(thr.round(4))

In [ ]:
# Confusion matrix: primary-blind (predict 1 on every signal) vs meta-filtered (predict only when p > p*).
# Source: oos_events_with_predictions.csv has the H1 2022 events + the model's calibrated proba.
ev = pd.read_csv(RESULTS / "oos_events_with_predictions.csv")
y = ev['label'].astype(int).to_numpy()
p = ev['calibrated_proba'].to_numpy()

P_STAR_DEFAULT = 0.5
meta_mask = p >= P_STAR_DEFAULT
primary_mask = np.ones(len(y), dtype=bool)

def conf(y_arr, mask):
    tp = int(((mask) & (y_arr == 1)).sum())
    fp = int(((mask) & (y_arr == 0)).sum())
    fn = int(((~mask) & (y_arr == 1)).sum())
    tn = int(((~mask) & (y_arr == 0)).sum())
    n_taken = int(mask.sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    return {'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
            'n_taken': n_taken, 'precision': round(precision, 4),
            'recall': round(recall, 4)}

summary = pd.DataFrame({
    'primary-blind (take every signal)': conf(y, primary_mask),
    f'meta-filtered (p >= {P_STAR_DEFAULT})': conf(y, meta_mask),
}).T
print("Dual confusion matrix — primary alone vs primary + meta filter (H1 2022 OOS):")
display(summary)

lift = (summary.loc[f'meta-filtered (p >= {P_STAR_DEFAULT})', 'precision']
        - summary.loc['primary-blind (take every signal)', 'precision'])
fp_avoided = (summary.loc['primary-blind (take every signal)', 'fp']
              - summary.loc[f'meta-filtered (p >= {P_STAR_DEFAULT})', 'fp'])
tp_missed = (summary.loc['primary-blind (take every signal)', 'tp']
             - summary.loc[f'meta-filtered (p >= {P_STAR_DEFAULT})', 'tp'])
print(f"\nPrecision lift from meta filter: {lift:+.4f}")
print(f"False positives avoided:         {fp_avoided}")
print(f"True positives missed:           {tp_missed}")

## Section 6 — Strategy construction (bonus, +10 marks)

Madmoun *Optional Session 3* recipe applied to the calibrated probabilities:
1. **Per-instrument Platt calibration** on CPCV OOF (slide 31).
2. **Bootstrap p\* = L / (G + L) gate** from train-fold TP/FP returns (slide 21).
3. **Six sizing functions** — `model_confidence`, `all_or_nothing`, `ncdf`, `linear_scaling`, `ecdf`, **`sops`** (Sharpe-Optimal Position Sizing, slide 34). Default ships **SOPS**.
4. **Causal EWMA σ̂** with `λ = 2 / (span + 1)`, `span = 100` (slide 39).
5. **Vol-targeted weight** `w = ŷ · σ_tgt / σ̂`, `σ_tgt = 10 %`, clipped to `[-10, +10]` (slide 40).
6. **Cross-sectional aggregation** `R_port = (1 / K_active) Σ w · r` (slide 41).
7. **Grinold-Kahn costs** — half-spread + linear impact charged on `|Δw|` per bar.

Four NN backbones are benchmarked head-to-head against SOPS (slides 45–53): **Linear** (DLinear-style), **LSTM**, **VLSTM**, **TFT**. Each is trained with the negative-annualised-Sharpe loss (slide 42) and Adam + early-stop on val Sharpe (slide 51).

In [ ]:
# Top-line strategy verdict.
metrics = pd.read_csv(RESULTS / "backtest_metrics.csv", index_col=0)
if metrics.shape[1] == 1:
    metrics.columns = ['value']
print("H1 2022 sealed-test backtest — vol-targeted SOPS variant (champion):")
display(metrics)

# Variant comparison head-to-head.
comp_path = RESULTS / "strategy_variant_comparison.csv"
if comp_path.exists():
    comp = pd.read_csv(comp_path)
    print("\nHead-to-head: SOPS vs 4 NN backbones (H1 2022 OOS):")
    display(comp.round(4))

In [ ]:
# Significance + deflation gate.
sig_path = RESULTS / "significance_summary.csv"
if sig_path.exists():
    sig = pd.read_csv(sig_path)
    print("Significance battery (R-13: at least four of five lenses agree on the verdict):")
    display(sig)

defl_path = RESULTS / "deflation_ladder.csv"
if defl_path.exists():
    defl = pd.read_csv(defl_path)
    print("\nDeflated Sharpe Ratio (DSR) at increasing N_eff:")
    display(defl)

## Section 7 — Deliverable CSV verification

Final sanity check on the two artefacts that the grader actually consumes.

In [ ]:
preds = pd.read_csv(OUTPUTS / "metamodel_predictions.csv")
wts = pd.read_csv(OUTPUTS / "strategy_weights.csv")

def audit(df: pd.DataFrame, name: str, value_col: str, value_range: tuple[float, float]) -> None:
    print(f"--- {name} ---")
    print(f"  rows: {len(df)}, columns: {list(df.columns)}")
    print(f"  date range: {df['date'].min()} -> {df['date'].max()}")
    n_dates = df['date'].nunique()
    n_inst = df['instrument'].nunique()
    print(f"  unique dates: {n_dates}, instruments: {n_inst}, expected: {n_dates*n_inst}")
    full_grid_ok = len(df) == n_dates * n_inst
    print(f"  full grid (no duplicates, no missing pairs): {full_grid_ok}")
    print(f"  {value_col} range: [{df[value_col].min():.4f}, {df[value_col].max():.4f}]")
    print(f"  value range respected ({value_range}): "
          f"{df[value_col].min() >= value_range[0] and df[value_col].max() <= value_range[1]}")
    print()

audit(preds, "metamodel_predictions.csv", "prediction", (0.0, 1.0))
audit(wts, "strategy_weights.csv", "weight", (-10.0, 10.0))

---

## H2 2022 rerun checklist

Per `SUBMISSION_README.md` §4:
1. Replace `data/ohlcv_data.csv` and `data/primary_signals.csv` with versions extended through Dec 2022.
2. The cleaned Bloomberg parquets cover the released window only. For H2 2022 we ship two macro inputs:
   - `data/features/f11_macro_context_oos.csv` (Part I-schema z-scored macro features, daily, Jul–Dec 2022)
   - `data/OOS_additional_data.xlsx` (raw H2 2022 macro source)
3. Re-run the pipeline. For models whose champion has a BBG-dependent feature, NaN-tolerant inference applies (see `bbg_missingness_ablation.csv`).
4. Re-emit the deliverable CSVs:
   ```bash
   uv run python scripts/build_submission_deliverables.py \
       --start 2022-07-01 --end 2022-12-31
   ```

## Limitations and caveats (R-list from `plan.md` §13)

- **R-10 (continuous-contract adjustment).** OHLCV is adjusted continuous-futures, not raw front-month. Strategy results are reported in this measurement frame; do **not** read them as raw-market WTI / S&P 500 P&L (see `plan.md` §11.5).
- **R-11 (BBG missingness in H2 2022).** The model uses Bloomberg-augmented features whose cleaned panel ends 2022-06-30. The H2 2022 rerun relies on NaN-tolerant inference paths; the documented AUC delta per asset class is in `bbg_missingness_ablation.csv`.
- **Per-instrument barrier geometry.** Six instruments (cl1s / fesx1s / ho1s / ng1s / nq1s / pl1s) use `h = 1` which converges to a next-day signed-return label. This is what the EDA argues for on those instruments (lag-1 peak in `corr(s_t, r_{t+1})`) but it is *not* a wide-window triple-barrier — readers should treat the label as 'did the next-bar return have the expected sign'.
